# 第14讲：文件后端与 SQLite 共享记忆
先观察原生 MemoryProvider 接口，再运行 A/B 与故障实验。OpenViking 是 [可选进阶](OPENVIKING.md)，不需要先安装它。

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
if not (here / 'native_notebook.py').exists():
    here = here / 'examples/14-external-memory-provider'
assert (here / 'native_notebook.py').exists()
sys.path.insert(0, str(here))
from native_notebook import run_hermes
ALLOW_MODEL = False
# GLM_API_KEY 与 GLM_BASE_URL 只通过启动进程的环境传入。


## 1. 直接调用原生接口（无费用）
本格展示 initialize、sync_turn 经 Manager 调度，以及自动召回和按需工具查询。输入是构造消息，不冒充模型回答。

In [ ]:
interface_run = run_hermes('''
import uuid
from run_provider import HERE, MemoryManager, FileBackedMemory
root = HERE / 'runtime/provider' / uuid.uuid4().hex
root.mkdir(parents=True, mode=0o700)
provider = FileBackedMemory(root / 'memory.json', identity='A')
manager = MemoryManager()
manager.add_provider(provider)
try:
    manager.initialize_all(session_id='notebook-A', hermes_home=str(root / 'A'))
    manager.sync_all('API 文档地址：docs.example.test/api-v7', '构造的确认文本', session_id='notebook-A')
    assert manager.flush_pending(timeout=20)
    assert not provider.pending()
    print(manager.prefetch_all('API 文档', session_id='notebook-A'))
    print(manager.handle_tool_call('memory_lookup', {'query': 'API'}))
finally:
    manager.shutdown_all()
''')
print('接口回执：', interface_run / 'process.log')


## 2. 两种后端运行同一实验
B 先启动，A 后写入；B 自动召回并按需查询。随后后端目录真正不可用，检查本地队列完整保留，恢复后重复补传不重复。默认仅验证接口与存储；真实模型链另在下一格运行。

In [ ]:
offline_run = run_hermes('''
from run_provider import exercise, FileBackedMemory
from run_shared_sqlite import SharedSQLiteMemory
exercise(FileBackedMemory)
exercise(SharedSQLiteMemory)
''')
print('两种后端回执：', offline_run / 'process.log')


## 3. 真实 Agent 链（会产生费用）
设置 ALLOW_MODEL=True 后，每种后端运行4轮对话。Client.turn 内直接调用 `agent.run_conversation(user_message=message, conversation_history=[])`，原生收尾链负责 sync_turn；不会把接口构造文本当作模型答案。下面先显示实际代码，再选择运行。

In [ ]:
source_run = run_hermes('''
import inspect
from run_provider import Client
print(inspect.getsource(Client.turn))
''')
print((source_run / 'process.log').read_text())
if ALLOW_MODEL:
    model_run = run_hermes('''
from run_provider import exercise, FileBackedMemory
from run_shared_sqlite import SharedSQLiteMemory
exercise(FileBackedMemory, with_model=True)
exercise(SharedSQLiteMemory, with_model=True)
''', timeout=900)
    print('真实调用记录：', model_run / 'process.log')
else:
    print('跳过付费模型调用')


## 核对与清理
查看 summary.json 的召回编号、工具查询和恢复结果；真实模式再逐项查看 A/B 的 turn-*.json。B 命中同一记录、B 回答正确、后端保存完整是不同检查。失败记录不能当成成功经验。每个实验用 finally 关闭 Manager/Provider/Agent；中断 Notebook 子进程时会先发送 SIGINT，保留时间执行清理。运行文件不删除，输出和密钥不公开。